In [1]:
# %% [markdown]
# # Urban Green Cover Thresholds for PM₂.₅ Mitigation: Baseline Predictive Models
# 
# **Stage:** Baseline Predictive ML (Stage 1)
# **Objective:** To understand and evaluate the predictive relationship between urban green cover and PM₂.₅ concentrations across Delhi NCR using a multimodal geospatial environmental dataset.
# 
# **IMPORTANT SCIENTIFIC WARNINGS:**
# 1. **Predictive vs. Causal:** Feature importances and coefficients generated in this notebook represent *predictive associations* only. They DO NOT establish causal pathways. For example, LST and NO₂ may act as mediators or colliders. Causal inference requires the downstream Double Machine Learning (DML) / Causal Forest stage.
# 2. **Spatial Extrapolation:** Performance metrics here evaluate within-domain predictive capability. They do not guarantee extrapolation to other cities.
# 3. **Temporal Generalization:** The period 2022-2025 has distinct meteorological regimes. Temporal cross-validation is used as a diagnostic, not a guarantee of future exact prediction.
# 4. **Static WorldCover:** ESA WorldCover 2021 is utilized as a static pre-study baseline land-cover composition. It does not represent land-cover changes during the 2022-2025 study period.

# %% [markdown]
# ## 1. Imports & Reproducibility
# %%
import os
import sys
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, median_absolute_error

# Enforce reproducibility
np.random.seed(42)
warnings.filterwarnings('ignore')

print(f"Python version: {sys.version}")
print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
import sklearn
print(f"scikit-learn version: {sklearn.__version__}")
import lightgbm
print(f"lightgbm version: {lightgbm.__version__}")

# %% [markdown]
# ## 2. Directory Setup
# %%
directories = [
    '../data/modeling_results_v2/metrics',
    '../data/modeling_results_v2/feature_importance',
    '../data/modeling_results_v2/validation',
    '../data/modeling_results_v2/predictions',
    '../data/modeling_results_v2/diagnostics',
    '../data/modeling_results_v2/figures',
    '../data/modeling_results_v2/models'
]
for d in directories:
    os.makedirs(d, exist_ok=True)
print("Output directories verified.")



Python version: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
pandas version: 3.0.5
numpy version: 2.5.1
scikit-learn version: 1.9.0
lightgbm version: 4.7.0
Output directories verified.


In [2]:
# %% [markdown]
# ## 3. Dataset Loading
# %%
v2_path = '../data/ml_ready/master_modeling_dataset_v2.csv'
train_path = '../data/modeling_final/train.csv'
test_path = '../data/modeling_final/test.csv'

df_v2 = pd.read_csv(v2_path)
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

print(f"Master Dataset Shape: {df_v2.shape}")
print(f"Train Dataset Shape: {df_train.shape}")
print(f"Test Dataset Shape: {df_test.shape}")

# %% [markdown]
# ## 4. Data Integrity Audit
# %%
audit_results = {}

# Row and column counts
assert df_v2.shape[0] == 1615, "V2 row count mismatch!"
assert df_train.shape[0] == 1292, "Train row count mismatch!"
assert df_test.shape[0] == 323, "Test row count mismatch!"
audit_results['v2_rows'] = int(df_v2.shape[0])
audit_results['train_rows'] = int(df_train.shape[0])
audit_results['test_rows'] = int(df_test.shape[0])

# Duplicate keys
duplicate_keys = df_v2.duplicated(subset=['station', 'year', 'month']).sum()
assert duplicate_keys == 0, "Duplicate station-year-month keys found!"
audit_results['duplicate_keys'] = int(duplicate_keys)

# Missing Target
missing_target = df_v2['pm25'].isna().sum()
assert missing_target == 0, "Missing values found in target pm25!"
audit_results['missing_pm25'] = int(missing_target)

# Infinite numeric values
inf_counts = np.isinf(df_v2.select_dtypes(include=np.number)).sum().sum()
assert inf_counts == 0, "Infinite values found in numeric features!"
audit_results['infinite_values'] = int(inf_counts)

print("Data Integrity Audit: PASS")

# %% [markdown]
# ## 5. Split Integrity Audit
# %%
# Train/Test schema identity
assert list(df_train.columns) == list(df_test.columns), "Train/Test schemas do not match!"

# Overlap Check
train_keys = set(zip(df_train['station'], df_train['year'], df_train['month']))
test_keys = set(zip(df_test['station'], df_test['year'], df_test['month']))
v2_keys = set(zip(df_v2['station'], df_v2['year'], df_v2['month']))

assert len(train_keys.intersection(test_keys)) == 0, "Train and Test sets overlap!"
assert train_keys.union(test_keys) == v2_keys, "Train + Test key universe does not match V2!"
audit_results['train_test_overlap'] = 0

# Representation Checks
for year in [2022, 2023, 2024, 2025]:
    assert year in df_train['year'].values, f"{year} missing from train set!"
    assert year in df_test['year'].values, f"{year} missing from test set!"

# IIT Delhi Condition
assert 'IIT_Delhi' not in df_test['station'].values, "IIT_Delhi found in test set!"
assert 'IIT_Delhi' in df_train['station'].values, "IIT_Delhi missing from train set!"

with open('../data/modeling_results_v2/validation/data_audit.json', 'w') as f:
    json.dump(audit_results, f, indent=4)
print("Split Integrity Audit: PASS")



Master Dataset Shape: (1615, 172)
Train Dataset Shape: (1292, 172)
Test Dataset Shape: (323, 172)
Data Integrity Audit: PASS
Split Integrity Audit: PASS


In [3]:
# %% [markdown]
# ## 6. Missingness Audit
# %%
missing_counts = df_v2.isna().sum()
missing_cols = missing_counts[missing_counts > 0]
missing_df = pd.DataFrame({
    'missing_count': missing_cols,
    'missing_percent': (missing_cols / len(df_v2)) * 100
}).sort_values('missing_percent', ascending=False)

print("Columns with missing values (to be median-imputed in pipeline):")
print(missing_df if not missing_df.empty else "No missing values found.")

# %% [markdown]
# ## 7. Feature Audit / Leakage Audit
# %%
leakage_candidates = [c for c in df_v2.columns if 'pm25' in c.lower() or 'pm_25' in c.lower() or 'target' in c.lower()]
print(f"Potential leakage columns identified: {leakage_candidates}")

# Predictors
exclude_cols = ['station'] + leakage_candidates
predictors = [c for c in df_train.columns if c not in exclude_cols]
target = 'pm25'

# Correlation diagnostic
correlations = df_train[predictors + [target]].select_dtypes(include=np.number).corr()[target].drop(target)
high_corr = correlations[abs(correlations) > 0.85]

leakage_audit = {
    'leakage_candidates': leakage_candidates,
    'excluded_from_predictors': exclude_cols,
    'highly_correlated_features': high_corr.to_dict()
}
with open('../data/modeling_results_v2/validation/leakage_audit.json', 'w') as f:
    json.dump(leakage_audit, f, indent=4)

if not high_corr.empty:
    print("WARNING: Highly correlated features found (correlation > 0.85). Review for potential physical leakage:")
    print(high_corr)

# Subsetting Data (Numeric only for baseline regressions)
numeric_predictors = df_train[predictors].select_dtypes(include=np.number).columns.tolist()

X_train = df_train[numeric_predictors]
y_train = df_train[target]
X_test = df_test[numeric_predictors]
y_test = df_test[target]

# Retain structures for Grouped CV
groups_station = df_train['station']
groups_year = df_train['year']

print(f"Final Predictor Count: {len(numeric_predictors)}")




Columns with missing values (to be median-imputed in pipeline):
No missing values found.
Potential leakage columns identified: ['pm25']
Final Predictor Count: 170


In [4]:
# %% [markdown]
# ## 8. Feature Group Definition
# %%
feature_groups = {
    'Green_Cover': [c for c in numeric_predictors if 'ndvi' in c.lower() or 'evi' in c.lower() or 'ndwi' in c.lower() or 'green' in c.lower()],
    'Meteorology': [c for c in numeric_predictors if 'era5' in c.lower() or 'lst' in c.lower() or 'temp' in c.lower() or 'rh' in c.lower() or 'wind' in c.lower() or 'blh' in c.lower()],
    'Pollution_Anthropogenic': [c for c in numeric_predictors if 'no2' in c.lower() or 's5p' in c.lower()],
    'Population': [c for c in numeric_predictors if 'worldpop' in c.lower()],
    'Road_Infrastructure': [c for c in numeric_predictors if 'road' in c.lower()],
    'Land_Cover': [c for c in numeric_predictors if 'worldcover' in c.lower()],
    'Spatial_Temporal': [c for c in numeric_predictors if c in ['latitude', 'longitude', 'year', 'month', 'season'] or 'month_sin' in c or 'month_cos' in c]
}

# %% [markdown]
# ## 9. Model Definitions
# %%
# Define models inside scikit-learn Pipelines
pipelines = {
    'Linear Regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LinearRegression())
    ]),
    'Ridge Regression': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', Ridge(random_state=42))
    ]),
    'Random Forest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1))
    ]),
    'LightGBM': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', LGBMRegressor(n_estimators=500, learning_rate=0.03, num_leaves=31, 
                                max_depth=8, min_child_samples=20, random_state=42, verbosity=-1))
    ])
}



In [5]:
# %% [markdown]
# ## 10. CV and 11. Model Training & Validation
# %%
def get_metrics(y_true, y_pred):
    return {
        'R2': r2_score(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAE': mean_absolute_error(y_true, y_pred),
        'MedianAE': median_absolute_error(y_true, y_pred)
    }

results = {}
test_predictions = pd.DataFrame({'station': df_test['station'], 'year': df_test['year'], 'month': df_test['month'], 'pm25_actual': y_test})

cv_primary = KFold(n_splits=5, shuffle=True, random_state=42)
cv_spatial = GroupKFold(n_splits=5)
cv_temporal = GroupKFold(n_splits=4)

grouped_cv_results = []

for name, pipe in pipelines.items():
    print(f"Training {name}...")
    
    # 1. Primary CV (Random 5-fold)
    cv_r2, cv_rmse, cv_mae = [], [], []
    for train_idx, val_idx in cv_primary.split(X_train):
        pipe.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        preds = pipe.predict(X_train.iloc[val_idx])
        m = get_metrics(y_train.iloc[val_idx], preds)
        cv_r2.append(m['R2']); cv_rmse.append(m['RMSE']); cv_mae.append(m['MAE'])
        
    # 2. Spatial Grouped CV
    spatial_r2 = []
    for train_idx, val_idx in cv_spatial.split(X_train, y_train, groups=groups_station):
        pipe.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        preds = pipe.predict(X_train.iloc[val_idx])
        spatial_r2.append(r2_score(y_train.iloc[val_idx], preds))
        
    # 3. Temporal Grouped CV
    temporal_r2 = []
    for train_idx, val_idx in cv_temporal.split(X_train, y_train, groups=groups_year):
        pipe.fit(X_train.iloc[train_idx], y_train.iloc[train_idx])
        preds = pipe.predict(X_train.iloc[val_idx])
        temporal_r2.append(r2_score(y_train.iloc[val_idx], preds))

    grouped_cv_results.append({
        'Model': name,
        'Spatial_CV_Mean_R2': np.mean(spatial_r2),
        'Temporal_CV_Mean_R2': np.mean(temporal_r2)
    })
    
    # 4. Final Train/Test Evaluation
    pipe.fit(X_train, y_train)
    joblib.dump(pipe, f'../data/modeling_results_v2/models/{name.lower().replace(" ", "_")}.joblib')
    
    train_preds = pipe.predict(X_train)
    test_preds = pipe.predict(X_test)
    test_predictions[f'pred_{name}'] = test_preds
    
    train_m = get_metrics(y_train, train_preds)
    test_m = get_metrics(y_test, test_preds)
    
    results[name] = {
        'Train R²': train_m['R2'],
        'Test R²': test_m['R2'],
        'Test RMSE': test_m['RMSE'],
        'Test MAE': test_m['MAE'],
        'Test MedianAE': test_m['MedianAE'],
        'CV R² mean': np.mean(cv_r2),
        'CV R² std': np.std(cv_r2),
        'CV RMSE mean': np.mean(cv_rmse),
        'CV RMSE std': np.std(cv_rmse),
        'CV MAE mean': np.mean(cv_mae),
        'CV MAE std': np.std(cv_mae),
        'R² gap': train_m['R2'] - test_m['R2']
    }

model_comp_df = pd.DataFrame(results).T.sort_values('Test R²', ascending=False)
model_comp_df.to_csv('../data/modeling_results_v2/metrics/model_comparison.csv')
pd.DataFrame(grouped_cv_results).to_csv('../data/modeling_results_v2/metrics/grouped_cv_performance.csv', index=False)
test_predictions.to_csv('../data/modeling_results_v2/predictions/test_predictions.csv', index=False)

display(model_comp_df)

# %% [markdown]
# ## 13. Year-wise Analysis & 14. Season-wise Analysis
# %%
# Derive Season
def get_season(month):
    if month in [12, 1, 2]: return 'Winter'
    elif month in [3, 4, 5, 6]: return 'Summer'
    elif month in [7, 8, 9]: return 'Monsoon'
    elif month in [10, 11]: return 'Post-monsoon'
    return 'Unknown'

df_test['season'] = df_test['month'].apply(get_season)

year_perf, season_perf = [], []

for name in pipelines.keys():
    # Year-wise
    for year in sorted(df_test['year'].unique()):
        mask = df_test['year'] == year
        y_t, y_p = y_test[mask], test_predictions.loc[mask, f'pred_{name}']
        m = get_metrics(y_t, y_p)
        year_perf.append({'Model': name, 'Year': year, 'N': mask.sum(), 'Mean_Actual': y_t.mean(), 
                          'Mean_Pred': y_p.mean(), 'Mean_Residual': (y_t - y_p).mean(), **m})
    # Season-wise
    for season in ['Winter', 'Summer', 'Monsoon', 'Post-monsoon']:
        mask = df_test['season'] == season
        y_t, y_p = y_test[mask], test_predictions.loc[mask, f'pred_{name}']
        m = get_metrics(y_t, y_p) if mask.sum() > 0 else {'R2': np.nan, 'RMSE': np.nan, 'MAE': np.nan, 'MedianAE': np.nan}
        season_perf.append({'Model': name, 'Season': season, 'N': mask.sum(), **m})

df_year_perf = pd.DataFrame(year_perf)
df_season_perf = pd.DataFrame(season_perf)

df_year_perf.to_csv('../data/modeling_results_v2/metrics/yearwise_performance.csv', index=False)
df_season_perf.to_csv('../data/modeling_results_v2/metrics/seasonwise_performance.csv', index=False)


Training Linear Regression...
Training Ridge Regression...
Training Random Forest...
Training LightGBM...


,Train R²,Test R²,Test RMSE,Test MAE,Test MedianAE,CV R² mean,CV R² std,CV RMSE mean,CV RMSE std,CV MAE mean,CV MAE std,R² gap
LightGBM,0.998917,0.897482,20.378313,10.460090,6.512763,0.946999,0.005428,15.633633,1.214987,10.265804,0.651181,0.101435
Random Forest,0.990701,0.875906,22.420377,12.243470,7.500420,0.924151,0.008008,18.692023,1.352753,12.073575,0.826129,0.114795
Linear Regression,0.876668,-1.714552,104.861556,26.275765,16.134595,0.814697,0.029495,29.069350,1.286629,22.011290,0.697223,2.591221
Ridge Regression,0.870734,-2.129229,112.586372,26.755592,15.665263,0.819724,0.019180,28.737494,0.658869,21.479842,0.914382,2.999962


In [6]:
# %% [markdown]
# ## 16. Feature Importance & 17. Feature Group Importance
# %%
feature_imp_records = {}

for name, pipe in pipelines.items():
    model_step = pipe.named_steps['model']
    if name in ['Linear Regression', 'Ridge Regression']:
        importance = model_step.coef_
        imp_df = pd.DataFrame({'Feature': numeric_predictors, 'Coefficient': importance, 'Abs_Importance': np.abs(importance)})
        imp_df = imp_df.sort_values('Abs_Importance', ascending=False)
        imp_df.to_csv(f'../data/modeling_results_v2/feature_importance/{name.lower().replace(" ", "_")}_coefficients.csv', index=False)
        feature_imp_records[name] = imp_df.set_index('Feature')['Abs_Importance']
    else:
        importance = model_step.feature_importances_
        imp_df = pd.DataFrame({'Feature': numeric_predictors, 'Importance': importance})
        imp_df = imp_df.sort_values('Importance', ascending=False)
        imp_df.to_csv(f'../data/modeling_results_v2/feature_importance/{name.lower().replace(" ", "_")}.csv', index=False)
        feature_imp_records[name] = imp_df.set_index('Feature')['Importance']

# Group Importance calculation
group_importance_df = pd.DataFrame(index=feature_groups.keys(), columns=pipelines.keys())
for model_name, imp_series in feature_imp_records.items():
    total_imp = imp_series.sum()
    for group_name, features in feature_groups.items():
        valid_features = [f for f in features if f in imp_series.index]
        group_sum = imp_series.loc[valid_features].sum()
        group_importance_df.loc[group_name, model_name] = (group_sum / total_imp) * 100 if total_imp > 0 else 0

print("\nFeature Group Importance (% of Total Predictive Importance):")
display(group_importance_df)

# %% [markdown]
# ## 18. Residual Diagnostics
# %%
best_model_name = model_comp_df.index[0]
best_preds = test_predictions[f'pred_{best_model_name}']
residuals = y_test - best_preds

res_summary = {
    'Mean_Residual': residuals.mean(),
    'Median_Residual': residuals.median(),
    'Residual_Std': residuals.std(),
    'Residual_Skewness': residuals.skew(),
    'Max_Underprediction': residuals.max(), # Large positive residual = actual > predicted
    'Max_Overprediction': residuals.min()   # Large negative residual = actual < predicted
}
pd.DataFrame([res_summary]).to_csv('../data/modeling_results_v2/diagnostics/residual_summary.csv', index=False)




Feature Group Importance (% of Total Predictive Importance):


,Linear Regression,Ridge Regression,Random Forest,LightGBM
Green_Cover,13.429332,25.34272,4.969528,46.055016
Meteorology,22.378509,18.049943,17.325999,23.068687
Pollution_Anthropogenic,32.483338,13.986067,1.29624,13.473892
Population,3.960801,2.565284,0.370474,0.601219
Road_Infrastructure,2.386078,5.187542,1.109386,2.643716
Land_Cover,12.166356,16.853458,1.420641,7.42876
Spatial_Temporal,7.193285,11.41541,72.58266,4.266183


In [7]:
# %% [markdown]
# ## 19. Required Visualizations
# %%
sns.set_theme(style="whitegrid")

# Figure 1: Actual vs Predicted PM2.5
plt.figure(figsize=(8, 8))
plt.scatter(y_test, best_preds, alpha=0.6, edgecolors='k')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.title(f'Fig 1: Actual vs Predicted PM₂.₅ ({best_model_name})')
plt.xlabel('Observed PM₂.₅')
plt.ylabel('Predicted PM₂.₅')
plt.text(0.05, 0.95, f"Test R²: {model_comp_df.loc[best_model_name, 'Test R²']:.3f}\nTest RMSE: {model_comp_df.loc[best_model_name, 'Test RMSE']:.1f}", 
         transform=plt.gca().transAxes, verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
plt.tight_layout()
plt.savefig('../data/modeling_results_v2/figures/fig1_actual_vs_predicted.png', dpi=300)
plt.close()

# Figure 2: Residual distribution
plt.figure(figsize=(8, 6))
sns.histplot(residuals, kde=True, bins=30, color='purple')
plt.axvline(0, color='black', linestyle='--')
plt.title(f'Fig 2: Residual Distribution ({best_model_name})')
plt.xlabel('Residual (Observed - Predicted)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.savefig('../data/modeling_results_v2/figures/fig2_residual_distribution.png', dpi=300)
plt.close()

# Figure 3: Residuals vs Predicted
plt.figure(figsize=(8, 6))
plt.scatter(best_preds, residuals, alpha=0.6, edgecolors='k')
plt.axhline(0, color='r', linestyle='--', lw=2)
plt.title(f'Fig 3: Residuals vs Predicted PM₂.₅ ({best_model_name})')
plt.xlabel('Predicted PM₂.₅')
plt.ylabel('Residual')
plt.tight_layout()
plt.savefig('../data/modeling_results_v2/figures/fig3_residuals_vs_predicted.png', dpi=300)
plt.close()

# Figure 4: Test performance by year
plt.figure(figsize=(10, 5))
sns.barplot(data=df_year_perf[df_year_perf['Model']==best_model_name], x='Year', y='RMSE', palette='Blues_d')
plt.title(f'Fig 4: Test RMSE by Year ({best_model_name})')
plt.ylabel('RMSE')
plt.tight_layout()
plt.savefig('../data/modeling_results_v2/figures/fig4_test_by_year.png', dpi=300)
plt.close()

# Figure 5: Test performance by season
plt.figure(figsize=(10, 5))
ax = sns.barplot(data=df_season_perf[df_season_perf['Model']==best_model_name], x='Season', y='RMSE', 
                 order=['Winter', 'Summer', 'Monsoon', 'Post-monsoon'], palette='Oranges_d')
plt.title(f'Fig 5: Test RMSE by Season ({best_model_name})')
plt.ylabel('RMSE')
for p, n in zip(ax.patches, df_season_perf[df_season_perf['Model']==best_model_name]['N']):
    ax.annotate(f'N={n}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', xytext=(0, 5), textcoords='offset points')
plt.tight_layout()
plt.savefig('../data/modeling_results_v2/figures/fig5_test_by_season.png', dpi=300)
plt.close()

# Figure 6: Spatial station performance (RMSE)
station_errors = test_predictions.copy()
station_errors['Residual_Sq'] = (station_errors['pm25_actual'] - station_errors[f'pred_{best_model_name}'])**2
station_rmse = station_errors.groupby('station')['Residual_Sq'].mean().apply(np.sqrt).reset_index(name='RMSE')

# Merge coordinates from df_v2
station_coords = df_v2[['station', 'latitude', 'longitude']].drop_duplicates()
station_rmse = station_rmse.merge(station_coords, on='station', how='left')

plt.figure(figsize=(10, 8))
sc = plt.scatter(station_rmse['longitude'], station_rmse['latitude'], 
                 c=station_rmse['RMSE'], cmap='Reds', s=100, edgecolor='k')
plt.colorbar(sc, label='RMSE')
plt.title('Fig 6: Spatial Station Performance Error (RMSE)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
for _, row in station_rmse.iterrows():
    if row['RMSE'] > station_rmse['RMSE'].quantile(0.85): # Label highest errors
        plt.text(row['longitude'], row['latitude'], row['station'], fontsize=8)
plt.tight_layout()
plt.savefig('../data/modeling_results_v2/figures/fig6_spatial_station_rmse.png', dpi=300)
plt.close()

# %% [markdown]
# ## 23. Research Findings & Anomalies
# %%
warnings_log = []

if model_comp_df.iloc[0]['Test R²'] < 0:
    warnings_log.append("CRITICAL: Best model test R² is negative.")
if any(model_comp_df['R² gap'] > 0.15):
    warnings_log.append("WARNING: Substantial overfitting detected (Train-Test R² gap > 0.15).")
if res_summary['Max_Underprediction'] > 150:
    warnings_log.append("WARNING: Extreme underprediction observed during high PM2.5 episodes.")

# Worst year/season
worst_year = df_year_perf[df_year_perf['Model']==best_model_name].sort_values('RMSE', ascending=False).iloc[0]
worst_season = df_season_perf[df_season_perf['Model']==best_model_name].sort_values('RMSE', ascending=False).iloc[0]

with open('../data/modeling_results_v2/validation/research_warnings.json', 'w') as f:
    json.dump({'warnings': warnings_log}, f, indent=4)

print("Baseline Summary Generated successfully.")



Baseline Summary Generated successfully.


In [8]:
# %% [markdown]
# ## Final Research Summary
# 
# **Best baseline model:** {best_model_name}
# **Best test R²:** {model_comp_df.iloc[0]['Test R²']:.3f}
# **Best test RMSE:** {model_comp_df.iloc[0]['Test RMSE']:.3f}
# **Best test MAE:** {model_comp_df.iloc[0]['Test MAE']:.3f}
# **Best CV R²:** {model_comp_df.iloc[0]['CV R² mean']:.3f}
# 
# **Worst-performing year:** {worst_year['Year']} (RMSE: {worst_year['RMSE']:.2f})
# **Worst-performing season:** {worst_season['Season']} (RMSE: {worst_season['RMSE']:.2f})
# 
# ### What these results DO support
# - **Predictive Validity:** Establishes the purely predictive limit of combined multi-scalar remote sensing and meteorological features.
# - **Feature Selection Insight:** Identifies which broad clusters (e.g., meteorology vs spatial infrastructure) carry the most predictive signal.
# - **Spatial-Temporal Error Characterization:** Pinpoints exactly where (stations) and when (seasons/years) baseline predictions systematically fail.
# 
# ### What these results DO NOT support
# - **Causal Effects:** Baseline predictive models DO NOT establish causal effects of green cover on PM₂.₅.
# - **Policy Intervention:** Feature importances cannot be used to estimate the PM₂.₅ reduction that would result from planting trees. LST, BLH, and NO₂ may mediate or confound these relationships.
# - **Direct Extrapolation:** Station-based performance in Delhi NCR does not universally guarantee the same predictive weights hold outside the NCR atmospheric regime.